# From Judging to Recommendation — Building a Protein Buying Guide
In a previous agentic design, we might have used a simple "judge" pattern. This would involve sending a broad question like "What is the best vegan protein?" to multiple large language models (LLMs), then using a separate “judge” agent to select the single best response. While useful, this approach can be limiting when a detailed comparison is needed.

To address this, we are shifting to a more powerful "synthesizer/improver" pattern for a very specific goal: to create a definitive buying guide for the best vegan protein powders available in the Netherlands. This requires more than just picking a single winner; it demands a detailed comparison based on strict criteria like clean ingredients, the absence of "protein spiking," and transparent amino acid profiles.

Instead of merely ranking responses, we will prompt a dedicated "synthesizer" agent to review all product recommendations from the other models. This agent will extract and compare crucial data points—ingredient lists, amino acid values, availability, and price—to build a single, improved report. This approach aims to combine the collective intelligence of multiple models to produce a guide that is richer, more nuanced, and ultimately more useful for a consumer than any individual response could be.


In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from anthropic import Anthropic
from IPython.display import Markdown, display

In [ ]:
load_dotenv(override=True)

In [ ]:
# Print the key prefixes to help with any debugging

openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if deepseek_api_key:
    print(f"DeepSeek API Key exists and begins {deepseek_api_key[:3]}")
else:
    print("DeepSeek API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

In [ ]:
# Protein Research: master prompt for the initial "teammate" LLMs.

request = (
    "Please research and identify the **Top 5 best vegan protein powders** available for purchase in the Netherlands. "
    "Your evaluation must be based on a comprehensive analysis of the following criteria, and you must present your findings as a ranked list from 1 to 5.\n\n"
    "**Evaluation Criteria:**\n\n"
    "1.  **No 'Protein Spiking':** The ingredients list must be clean. Avoid products with 'AMINO MATRIX' or similar proprietary blends designed to inflate protein content.\n\n"
    "2.  **Transparent Amino Acid Profile:** Preference should be given to brands that disclose a full amino acid profile, with high EAA and Leucine content.\n\n"
    "3.  **Sweetener & Sugar Content:** Scrutinize the ingredient list for all sugars and artificial sweeteners. For each product, you must **list all identified sweeteners** (e.g., sucralose, stevia, erythritol, aspartame, sugar).\n\n"
    "4.  **Taste Evaluation from Reviews:** You must search for and analyze customer reviews on Dutch/EU e-commerce sites (like Body & Fit, bol.com, etc.). "
    "Summarize the general consensus on taste. Specifically look for strong positive reviews and strong negative reviews using keywords like 'delicious', 'great taste', 'bad', 'awful', 'impossible to swallow', or 'tastes like cardboard'.\n\n"
    "5.  **Availability in the Netherlands:** The products must be easily accessible to Dutch consumers.\n\n"
    "**Required Output Format:**\n"
    "For each of the Top 5 products, please provide:\n"
    "- **Rank (1-5)**\n"
    "- **Brand Name & Product Name**\n"
    "- **Justification:** A summary of why it's a top product based on protein quality (Criteria 1 & 2).\n"
    "- **Listed Sweeteners:** The list of sugar/sweetener ingredients you found.\n"
    "- **Taste Review Summary:** The summary of your findings from customer reviews."
)

request += "Answer only with the question, no explanation."
messages = [{"role": "user", "content": request}]

In [ ]:
messages

In [ ]:
openai = OpenAI(api_key=google_api_key, base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
response = openai.chat.completions.create(
    model="gemini-2.5-flash",
    messages=messages,
)
question = response.choices[0].message.content
print(question)


In [ ]:
teammates = []
answers = []
messages = [{"role": "user", "content": question}]

In [ ]:
# The API we know well

model_name = "gpt-4o-mini"

response = openai.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
teammates.append(model_name)
answers.append(answer)

In [ ]:
# Anthropic has a slightly different API, and Max Tokens is required

model_name = "claude-3-7-sonnet-latest"

claude = Anthropic()
response = claude.messages.create(model=model_name, messages=messages, max_tokens=1000)
answer = response.content[0].text

display(Markdown(answer))
teammates.append(model_name)
answers.append(answer)

In [ ]:
gemini = OpenAI(api_key=google_api_key, base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
model_name = "gemini-2.0-flash"

response = gemini.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
teammates.append(model_name)
answers.append(answer)

In [ ]:
deepseek = OpenAI(api_key=deepseek_api_key, base_url="https://api.deepseek.com/v1")
model_name = "deepseek-chat"

response = deepseek.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
teammates.append(model_name)
answers.append(answer)

In [ ]:
groq = OpenAI(api_key=groq_api_key, base_url="https://api.groq.com/openai/v1")
model_name = "llama-3.3-70b-versatile"

response = groq.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
teammates.append(model_name)
answers.append(answer)

# Calling Ollama now

In [ ]:
!ollama pull llama3.2

In [ ]:
ollama = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')
model_name = "llama3.2"

response = ollama.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
teammates.append(model_name)
answers.append(answer)

In [ ]:
# So where are we?

print(teammates)
print(answers)

In [ ]:
# It's nice to know how to use "zip"
for teammate, answer in zip(teammates, answers):
    print(f"Teammate: {teammate}\n\n{answer}")

In [ ]:
# Let's bring this together - note the use of "enumerate"

together = ""
for index, answer in enumerate(answers):
    together += f"# Response from teammate {index+1}\n\n"
    together += answer + "\n\n"

In [ ]:
print(together)

In [ ]:
# The `question` variable would hold the content of the `request` from Step 1.
# The `teammates` variable would be a list of the responses from the other LLMs.

# This `formatter` prompt would then be sent to your final synthesizer LLM.
formatter = f"""You are a discerning Health and Nutrition expert creating a definitive consumer guide. You have received {len(teammates)} 'Top 5' lists from different AI assistants based on the following detailed request:

---
**Original Request:**
"{question}"
---

Your task is to synthesize these lists into a single, master "Top 5 Vegan Proteins in the Netherlands" report. You must critically evaluate the provided information, resolve any conflicts, and create a final ranking based on a holistic view.

**Your synthesis and ranking logic must follow these rules:**
1.  **Taste is a priority:** Products with consistently poor taste reviews (e.g., described as 'bad', 'undrinkable', 'cardboard') must be ranked lower or disqualified, even if their nutritional profile is excellent. Highlight products praised for their good taste.
2.  **Low sugar scores higher:** Products with fewer or no artificial sweeteners are superior. A product sweetened only with stevia is better than one with sucralose and acesulfame-K. Unsweetened products should be noted as a top choice for health-conscious consumers.
3.  **Evidence over claims:** Base your ranking on the evidence provided by the assistants (ingredient lists, review summaries). Note any consensus between the assistants, as this indicates a stronger recommendation.

**Required Report Structure:**
1.  **Title:** "The Definitive Guide: Top 5 Vegan Proteins in the Netherlands".
2.  **Introduction:** Briefly explain the methodology, mentioning that the ranking is based on protein quality, low sugar, and real-world taste reviews.
3.  **The Top 5 Ranking:** Present the final, synthesized list from 1 to 5. For each product:
    - **Rank, Brand, and Product Name.**
    - **Synthesized Verdict:** A summary paragraph explaining its final rank. This must include:
        - **Protein Quality:** A note on its ingredients and amino acid profile.
        - **Sweetener Profile:** A comment on its sweetener content and why that's good or bad.
        - **Taste Consensus:** The final verdict on its taste based on the review analysis. (e.g., "While nutritionally sound, it ranks lower due to consistent complaints about its chalky taste, as noted by Assistants 1 and 3.")
4.  **Honorable Mentions / Products to Avoid:** Briefly list any products that appeared in the lists but didn't make the final cut, and state why (e.g., "Product X was disqualified due to multiple artificial sweeteners and poor taste reviews.").
"""

In [ ]:
print(formatter)

In [ ]:
formatter_messages = [{"role": "user", "content": formatter}]

In [ ]:
openai = OpenAI(api_key=google_api_key, base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
response = openai.chat.completions.create(
    model="gemini-2.5-flash",
    messages=formatter_messages,
)
results = response.choices[0].message.content
display(Markdown(results))